# Notebook 04: Feature Extraction

**Objective:**
Extract 30+ physical and spectral features from the 10-second signals and build the final ML dataset.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

project_root = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root) # Insert at front to guarantee local load

# IMPORT V2 TO BYPASS JUPYTER KERNEL CACHING!
from src.feature_extraction_v2 import extract_features

PROCESSED_DIR = os.path.join(project_root, 'processed_data')
OUTPUT_DIR = os.path.join(project_root, 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
df_labels = pd.read_csv(os.path.join(PROCESSED_DIR, 'labels_3class.csv'))
signals_loaded = np.load(os.path.join(PROCESSED_DIR, 'preprocessed_signals.npz'))

features_list = []

for index, row in tqdm(df_labels.iterrows(), total=len(df_labels), desc="Extracting Features"):
    rec_id = str(row['ID'])
    if rec_id in signals_loaded:
        sig = signals_loaded[rec_id]
        
        feat_dict = extract_features(sig, fs=30)
        feat_dict['ID'] = rec_id
        feat_dict['SQI_Label'] = row['SQI_Label']
        
        features_list.append(feat_dict)

features_df = pd.DataFrame(features_list).fillna(0)
out_path = os.path.join(OUTPUT_DIR, 'extracted_features.csv')
features_df.to_csv(out_path, index=False)
print(f"\nSaved features to {out_path}")

Extracting Features: 100%|██████████| 3888/3888 [00:09<00:00, 418.61it/s]



Saved features to c:\Users\rajka\PPG-IIIT\outputs\extracted_features.csv


### Validation Check (Crucial)

In [ ]:
feature_cols = [c for c in features_df.columns if c not in ['ID', 'SQI_Label']]
print("Number of Total Columns:", len(features_df.columns))
print("Number of True ML Features:", len(feature_cols))
print("Data Shape:", features_df.shape)
print("\nFeature List:")
print(feature_cols)
print("\nHead:")
print(features_df.head())

Number of Total Columns: 32
Number of True ML Features: 30
Data Shape: (3888, 32)

Feature List:
['mean', 'median', 'variance', 'std_dev', 'rms', 'mad', 'iqr', 'range', 'cv', 'kurtosis', 'skewness', 'energy', 'zcr', 'snr', 'peak_count', 'mean_peak_interval', 'std_peak_interval', 'rmssd', 'mean_peak_height', 'std_peak_height', 'mean_prominence', 'std_prominence', 'mean_peak_width', 'std_peak_width', 'dominant_freq', 'spectral_entropy', 'power_low', 'power_mid', 'power_high', 'power_ratio']

Head:
           mean    median  variance  std_dev  rms       mad       iqr  \
0  2.664535e-17 -0.000896       1.0      1.0  1.0  0.654920  1.303975   
1 -1.554312e-17 -0.180528       1.0      1.0  1.0  0.752965  1.484400   
2 -1.184238e-17 -0.056860       1.0      1.0  1.0  0.675837  1.354323   
3  5.921189e-18 -0.261415       1.0      1.0  1.0  0.574165  1.251441   
4 -1.924387e-17 -0.164570       1.0      1.0  1.0  0.759552  1.556199   

      range            cv  kurtosis  ...  mean_peak_width  s